# SauberModel5 — اختبار بيانات جديدة (v2)

**التحسينات مقارنةً بالنسخة الأولى:**
| التغيير | السبب |
|---|---|
| `squeeze(1)` على outputs | توحيد الأبعاد مع preds |
| التحقق من `class_to_idx` | ضمان NORMAL=0, PNEUMONIA=1 |
| `num_workers=4` | تسريع تحميل البيانات |
| حساب Recall و F1 | المقاييس الطبية الأهم |
| دعم `best_threshold` | تجاوز حد 0.5 الافتراضي |

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# ── تعريف مكونات الموديل (مطابق 100% لـ Cell 4 في ملف التدريب) ──────────
class SELayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SELayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class DropPath(nn.Module):
    def __init__(self, p: float = 0.0):
        super().__init__()
        self.p = p
    def forward(self, x):
        if not self.training or self.p == 0.0: return x
        keep  = 1.0 - self.p
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask  = torch.empty(shape, dtype=x.dtype, device=x.device).bernoulli_(keep) / keep
        return x * mask

class InnerResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, k_size=3, is_forth=False, drop2d_p=0.4, drop_path_p=0.2):
        super().__init__()
        if is_forth:
            self.conv = nn.Sequential(
                nn.Conv2d(in_c, out_c, (3, 1), padding='same'), nn.ReLU(),
                nn.Conv2d(out_c, out_c, (1, 3), padding='same'), nn.ReLU())
        else:
            self.conv = nn.Sequential(nn.Conv2d(in_c, out_c, k_size, padding='same'), nn.ReLU())
        self.bn        = nn.BatchNorm2d(out_c)
        self.drop2d    = nn.Dropout2d(p=drop2d_p)
        self.drop_path = DropPath(p=drop_path_p)
        self.shortcut  = nn.Sequential(nn.Conv2d(in_c, out_c, 1), nn.BatchNorm2d(out_c)) if in_c != out_c else nn.Identity()
    def forward(self, x):
        res = self.drop_path(self.drop2d(self.bn(self.conv(x))))
        return res + self.shortcut(x)

class SauberModel5(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem        = nn.Sequential(nn.Conv2d(3, 64, 3, padding='same'), nn.ReLU())
        self.b1          = nn.ModuleList([InnerResidualBlock(64 if i==0 else 96,  96,  3) for i in range(3)])
        self.b2          = nn.ModuleList([InnerResidualBlock(64 if i==0 else 32,  32,  5) for i in range(3)])
        self.b3          = nn.ModuleList([InnerResidualBlock(64 if i==0 else 128, 128, 3) for i in range(3)])
        self.b4          = nn.ModuleList([InnerResidualBlock(64 if i==0 else 64,  64, is_forth=True) for i in range(3)])
        self.se          = SELayer(320)
        self.concat_conv = nn.Sequential(nn.Conv2d(320, 128, 1), nn.BatchNorm2d(128))
        self.dropout     = nn.Dropout(0.3)
        self.id_proj     = nn.Sequential(nn.Conv2d(64, 128, 1), nn.BatchNorm2d(128))
        self.gap         = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Sequential(nn.Flatten(), nn.Linear(128, 1))
    def forward(self, x):
        x        = self.stem(x)
        id_outer = x
        out1, out2, out3, out4 = x, x, x, x
        for i in range(3):
            out1, out2, out3, out4 = (self.b1[i](out1), self.b2[i](out2), self.b3[i](out3), self.b4[i](out4))
        m = torch.cat([out1, out2, out3, out4], dim=1)
        m = self.se(m)
        m = self.dropout(self.concat_conv(m))
        m = self.gap(m + self.id_proj(id_outer))
        return self.classifier(m)

print('تم تعريف الموديل بنجاح ✓')

In [ ]:
# ── تحميل الأوزان ─────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'الجهاز: {device}')

model_path = r'D:\Pnuomonia\Pneumonia\SauberModel5_SOTA_E50_A93.77_R98.21_20260317_0445.pth'

model = SauberModel5().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()  # يوقف Dropout و BatchNorm

print(f'تم تحميل الموديل بنجاح ✓')
print(f'عدد المعاملات: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ── إعداد البيانات ────────────────────────────────────────────────────────
#
# هيكل المجلد المطلوب:
#   new_data_path/
#       NORMAL/       ← صور الأشخاص الأصحاء
#       PNEUMONIA/    ← صور الالتهاب الرئوي
#
# ⚠️ أسماء المجلدات حساسة لحالة الأحرف على Linux

new_data_path = r'C:\Path\To\New\Data'  # ← استبدليه بمسار بياناتك

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

new_dataset = datasets.ImageFolder(new_data_path, transform=test_transform)
new_loader  = DataLoader(
    new_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,           # تسريع تحميل البيانات
    persistent_workers=True,
    prefetch_factor=4,
)

# ── فحص التسميات (خطوة إجبارية) ──────────────────────────────────────────
print('class_to_idx:', new_dataset.class_to_idx)
# المتوقع دائماً: {'NORMAL': 0, 'PNEUMONIA': 1}
# إذا ظهر العكس → أسماء المجلدات خاطئة → النتائج ستكون مقلوبة

assert new_dataset.class_to_idx.get('NORMAL') == 0,    'خطأ: مجلد NORMAL غير موجود أو ترتيبه خاطئ'
assert new_dataset.class_to_idx.get('PNEUMONIA') == 1, 'خطأ: مجلد PNEUMONIA غير موجود أو ترتيبه خاطئ'

import collections
counts = collections.Counter(new_dataset.targets)
print(f'NORMAL   : {counts[0]:,} صورة')
print(f'PNEUMONIA: {counts[1]:,} صورة')
print(f'الإجمالي : {len(new_dataset):,} صورة  |  {len(new_loader)} batch')

In [ ]:
# ── حلقة التقييم ──────────────────────────────────────────────────────────
#
# يمكنك تغيير THRESHOLD إلى best_threshold من Cell 9 في ملف التدريب
# بدلاً من الاعتماد على 0.5 الافتراضي
THRESHOLD = 0.5  # ← غيّريه إذا لديكِ best_threshold من ملف التدريب

all_probs  = []
all_labels = []

with torch.no_grad():
    for imgs, labels in new_loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()

        # squeeze(1): يحوّل [batch, 1] → [batch] لتوحيد الأبعاد مع labels
        logits = model(imgs).squeeze(1)
        probs  = torch.sigmoid(logits)

        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

all_probs  = torch.cat(all_probs)
all_labels = torch.cat(all_labels)
preds      = (all_probs >= THRESHOLD).float()

# ── حساب المقاييس ──────────────────────────────────────────────────────────
N  = len(all_labels)
TP = int(((preds == 1) & (all_labels == 1)).sum())
TN = int(((preds == 0) & (all_labels == 0)).sum())
FP = int(((preds == 1) & (all_labels == 0)).sum())
FN = int(((preds == 0) & (all_labels == 1)).sum())

accuracy  = (TP + TN) / N
recall    = TP / (TP + FN + 1e-8)        # الأهم طبياً — لا نريد تفويت مريض
precision = TP / (TP + FP + 1e-8)
f1        = 2 * precision * recall / (precision + recall + 1e-8)
specificity = TN / (TN + FP + 1e-8)     # حساسية اكتشاف الأصحاء

print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'النتائج على البيانات الجديدة  (threshold={THRESHOLD})')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'العينات الكلية : {N}')
print(f'Accuracy       : {accuracy*100:.2f}%   ({TP+TN}/{N} صحيح)')
print(f'Recall         : {recall*100:.2f}%   ({TP}/{TP+FN} مريض تم اكتشافه)')
print(f'Precision      : {precision*100:.2f}%')
print(f'F1 Score       : {f1*100:.2f}%')
print(f'Specificity    : {specificity*100:.2f}%')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Confusion Matrix:')
print(f'                 Pred NORMAL  Pred PNEUMONIA')
print(f'True NORMAL         {TN:>4}           {FP:>4}')
print(f'True PNEUMONIA      {FN:>4}           {TP:>4}')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

In [ ]:
# ── (اختياري) إيجاد أفضل threshold على البيانات الجديدة ───────────────────
import numpy as np
import matplotlib.pyplot as plt

probs_np  = all_probs.numpy().astype(np.float32)
labels_np = all_labels.numpy().astype(np.float32)

thresholds = np.linspace(0, 1, 201, dtype=np.float32)
f1s, recalls, precisions, accs = [], [], [], []

for thr in thresholds:
    p   = (probs_np >= thr).astype(np.int8)
    lbl = labels_np.astype(np.int8)
    TP_ = int(((p == 1) & (lbl == 1)).sum())
    TN_ = int(((p == 0) & (lbl == 0)).sum())
    FP_ = int(((p == 1) & (lbl == 0)).sum())
    FN_ = int(((p == 0) & (lbl == 1)).sum())
    pr  = TP_ / (TP_ + FP_ + 1e-8)
    rc  = TP_ / (TP_ + FN_ + 1e-8)
    precisions.append(pr)
    recalls.append(rc)
    f1s.append(2 * pr * rc / (pr + rc + 1e-8))
    accs.append((TP_ + TN_) / len(labels_np))

best_idx = int(np.argmax(f1s))
best_thr = float(thresholds[best_idx])
print(f'أفضل threshold (max F1 على البيانات الجديدة): {best_thr:.3f}')
print(f'F1={f1s[best_idx]*100:.2f}%  Recall={recalls[best_idx]*100:.2f}%  Precision={precisions[best_idx]*100:.2f}%')

plt.figure(figsize=(10, 5))
plt.plot(thresholds, accs,       label='Accuracy',  color='steelblue')
plt.plot(thresholds, recalls,    label='Recall',    color='tomato')
plt.plot(thresholds, precisions, label='Precision', color='seagreen')
plt.plot(thresholds, f1s,        label='F1',        color='orange', lw=2)
plt.axvline(best_thr, color='gray', linestyle='--', label=f'Best thr={best_thr:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Metrics vs Threshold — البيانات الجديدة')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()